# Análise de clientes de um distribuidor atacadista com SQL

Usei SQL para responder 3 perguntas que um diretor comercial faria sobre os clientes de um distribuidor: onde está o dinheiro, quem são os clientes que mais pesam e se a empresa depende de poucos deles.

**Resposta curta:** o dinheiro está em produtos frescos e mercearia; o cliente de varejo vale mais que o de restaurante e hotel; e nenhum cliente sozinho ameaça o negócio.

**Dados:** [Wholesale Customers (Kaggle)](https://www.kaggle.com/datasets/smitisinghal/wholesale-customers-data), originalmente do repositório UCI. Cada linha é um cliente, com o valor comprado no ano em 6 categorias (Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen), o canal e a região.

**Glossário:** *Horeca* = Hotéis, Restaurantes e Cafés. *Varejo* = mercados e lojas.

**Abaixo de cada pergunta, em itálico, deixo a minha leitura do resultado e o impacto no negócio.**



## Preparação: carregar o CSV no banco SQLite
Os códigos de canal e região viram nomes em colunas novas (`canal`, `regiao`); as colunas originais com números continuam na tabela.

In [1]:
import sqlite3, pandas as pd

df = pd.read_csv('/content/Wholesale customers data.csv')
df['canal']  = df['Channel'].map({1: 'Horeca', 2: 'Varejo'})
df['regiao'] = df['Region'].map({1: 'Lisboa', 2: 'Porto', 3: 'Outras'})

conn = sqlite3.connect('wholesale.db')
df.to_sql('clientes', conn, index=False, if_exists='replace')
pd.read_sql("SELECT * FROM clientes LIMIT 5", conn)

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,canal,regiao
0,2,3,12669,9656,7561,214,2674,1338,Varejo,Outras
1,2,3,7057,9810,9568,1762,3293,1776,Varejo,Outras
2,2,3,6353,8808,7684,2405,3516,7844,Varejo,Outras
3,1,3,13265,1196,4221,6404,507,1788,Horeca,Outras
4,2,3,22615,5410,7198,3915,1777,5185,Varejo,Outras


## P1 — Quantos clientes existem em cada canal?

In [2]:
pd.read_sql("""
SELECT canal, COUNT(*) AS qtd_clientes
FROM clientes
GROUP BY canal
""", conn)

,canal,qtd_clientes
0,Horeca,298
1,Varejo,142


*Apesar de o Horeca ter mais que o dobro de clientes do Varejo (298 x 142), isso não significa que tenha mais peso no negócio. Precisamos ver quanto cada canal compra (P3).*

## P2 — Qual o faturamento total em cada uma das 6 categorias?

In [3]:
pd.read_sql("""
SELECT SUM(Fresh) AS total_fresh, SUM(Milk) AS total_milk,
       SUM(Grocery) AS total_grocery, SUM(Frozen) AS total_frozen,
       SUM(Detergents_Paper) AS total_detergents, SUM(Delicassen) AS total_delicassen
FROM clientes
""", conn)

,total_fresh,total_milk,total_grocery,total_frozen,total_detergents,total_delicassen
0,5280131,2550357,3498562,1351650,1267857,670943


*Fresh e Grocery concentram 60% do faturamento (36% e 24%). É onde a logística e o estoque precisam de mais atenção. Próxima pergunta: os dados não trazem margem; se as categorias menores tiverem lucro maior, vale direcionar o comercial para elas com mais intensidade.*

## P3 — Quanto cada cliente compra, em média, de Grocery em cada canal?

In [4]:
pd.read_sql("""
SELECT canal, ROUND(AVG(Grocery), 0) AS media_grocery
FROM clientes
GROUP BY canal
""", conn)

,canal,media_grocery
0,Horeca,3962.0
1,Varejo,16323.0


*Isso responde à P1: mesmo com mais que o dobro dos clientes no Horeca, cada cliente do Varejo compra em média 4,1x mais em Grocery. Ou seja, são menos clientes, porém mais robustos, e merecem atenção diferenciada. Perder um cliente do Varejo equivale a perder 4 do Horeca.*

## P4 — Quais são os 10 clientes que mais compram no total?

In [5]:
pd.read_sql("""
SELECT rowid AS cliente,
       Fresh + Milk + Grocery + Frozen + Detergents_Paper + Delicassen AS faturamento
FROM clientes
ORDER BY faturamento DESC
LIMIT 10
""", conn)

,cliente,faturamento
0,86,199891
1,48,192714
2,182,190169
3,184,185683
4,62,165881
5,87,150497
6,326,130877
7,334,120291
8,24,110829
9,212,107155


*Os 10 maiores clientes (2,3% da base) somam juntos 10,6% do faturamento, e o maior deles representa 1,4%. Em outras palavras, a empresa não tem grande dependência de poucos clientes.*

## P5 — Quantos clientes do Varejo em Lisboa compram mais de 10.000 em Grocery?

In [6]:
pd.read_sql("""
SELECT COUNT(*) AS total_varejo_lisboa,
       SUM(CASE WHEN Grocery > 10000 THEN 1 ELSE 0 END) AS acima_10mil
FROM clientes
WHERE canal = 'Varejo' AND regiao = 'Lisboa'
""", conn)

,total_varejo_lisboa,acima_10mil
0,18,14


*Em Lisboa, 14 dos 18 clientes do Varejo (78%) compram mais de 10 mil em Grocery. Ou seja, como vimos na P3, Grocery é o carro-chefe do Varejo, e garantir variedade e estoque dessa categoria em Lisboa é fundamental pra manter esses clientes.*

## P6 — Quantos clientes e quanto faturamento há em cada porte?

- **Grande:** compra total de 50.000 ou mais
- **Médio:** de 20.000 a 49.999
- **Pequeno:** abaixo de 20.000

In [7]:
pd.read_sql("""
SELECT CASE WHEN (Fresh + Milk + Grocery + Frozen + Detergents_Paper + Delicassen) >= 50000 THEN 'Grande'
            WHEN (Fresh + Milk + Grocery + Frozen + Detergents_Paper + Delicassen) >= 20000 THEN 'Médio'
            ELSE 'Pequeno'
       END AS porte,
       COUNT(*) AS qtd_clientes,
       SUM(Fresh + Milk + Grocery + Frozen + Detergents_Paper + Delicassen) AS faturamento
FROM clientes
GROUP BY porte
""", conn)

,porte,qtd_clientes,faturamento
0,Grande,66,5215783
1,Médio,228,7446613
2,Pequeno,146,1957104


*A maior fatia do faturamento está nos clientes de médio porte: 52% dos clientes e 51% do faturamento. Já os 66 clientes Grandes (15%) faturam 5,2 mi, 70% do que os 228 Médios faturam. Ou seja, perder um cliente Grande pesa em média 2,4x mais que perder um Médio; é o grupo que merece atenção prioritária.*

## P7 — Quais clientes compram Fresh acima da média geral?

In [8]:
pd.read_sql("""
SELECT rowid AS cliente, Fresh
FROM clientes
WHERE Fresh > (SELECT AVG(Fresh) FROM clientes)
ORDER BY Fresh DESC
""", conn)

,cliente,Fresh
0,182,112151
1,126,76237
2,285,68951
3,40,56159
4,259,56083
...,...,...
153,282,12238
154,123,12212
155,78,12205
156,7,12126


*158 clientes (36%) compram Fresh acima da média de 12 mil. Se a média representasse o cliente típico, seriam perto da metade. Ou seja, poucos clientes muito grandes (o maior compra 9x a média) puxam a média para cima, e ela superestima o cliente comum: para metas e estoque por cliente, a média não é uma boa referência.*

## P8 — Em cada canal, quem são os clientes que mais compram Fresh?

In [9]:
pd.read_sql("""
SELECT rowid AS cliente, canal, Fresh,
       RANK() OVER (PARTITION BY canal ORDER BY Fresh DESC) AS posicao
FROM clientes
ORDER BY canal, posicao
""", conn)

,cliente,canal,Fresh,posicao
0,182,Horeca,112151,1
1,126,Horeca,76237,2
2,285,Horeca,68951,3
3,40,Horeca,56159,4
4,259,Horeca,56083,5
...,...,...,...,...
435,354,Varejo,117,138
436,66,Varejo,85,139
437,358,Varejo,37,140
438,97,Varejo,23,141


*No Horeca, o maior comprador de Fresh (cliente 182, 112 mil) compra 1,5x o segundo colocado. No Varejo, o líder é o cliente 48 (44 mil). Ou seja, os maiores compradores de Fresh estão no Horeca, o que faz sentido para restaurantes e hotéis, que usam produto fresco todos os dias.*

## P9 — Quanto cada cliente representa (%) do Fresh do seu canal?

In [10]:
pd.read_sql("""
SELECT rowid AS cliente, canal, Fresh,
       ROUND(Fresh * 100.0 / SUM(Fresh) OVER (PARTITION BY canal), 2) AS pct_canal
FROM clientes
ORDER BY pct_canal DESC
""", conn)

,cliente,canal,Fresh,pct_canal
0,48,Varejo,44466,3.52
1,53,Varejo,40721,3.22
2,371,Varejo,39679,3.14
3,62,Varejo,35942,2.84
4,182,Horeca,112151,2.79
...,...,...,...,...
435,356,Horeca,190,0.00
436,413,Horeca,97,0.00
437,97,Varejo,23,0.00
438,219,Varejo,18,0.00


*Nenhum cliente passa de 3,6% do Fresh do seu canal: o 48 é o mais relevante no Varejo (3,5%) e o 182 no Horeca (2,8%). Ou seja, mesmo dentro de cada canal não existe dependência de um cliente só, o que confirma a P4.*

## Principais achados
- **Contagem engana:** o Horeca tem o dobro de clientes, mas cada cliente do Varejo compra 4,1x mais em Grocery (P1, P3).
- **Faturamento concentrado em 2 categorias:** Fresh e Grocery somam 60% (P2).
- **Carteira bem distribuída:** os 10 maiores clientes somam só 10,6% do faturamento e nenhum cliente passa de 3,6% do seu canal; o risco de dependência é baixo (P4, P9).
- **Os clientes Médios sustentam metade do faturamento**, e cada Grande vale em média 2,4x um Médio (P6).
- **A média não descreve o cliente típico:** poucos clientes muito grandes puxam a média para cima (P7).


*Sintaxe com apoio de guias pessoais e IA; análise e conclusões são minhas.*